In [3]:
import timeit

# Traditional for loop approach
def square_numbers_for_loop(n):
    squares = []
    for x in range(n):
        squares.append(x**2)
    return squares

# List comprehension approach
def square_numbers_list_comprehension(n):
    return [x**2 for x in range(n)]

# Number of elements in the list
n = 10**8

# Time the for loop approach
time_for_loop = timeit.timeit('square_numbers_for_loop(n)', 
                              globals=globals(), 
                              number=10)

# Time the list comprehension approach
time_list_comprehension = timeit.timeit('square_numbers_list_comprehension(n)', 
                                        globals=globals(), 
                                        number=10)

# Print the results
print(f"Time for traditional for loop: {time_for_loop:.6f} seconds")
print(f"Time for list comprehension: {time_list_comprehension:.6f} seconds")

# Compare the speed
speedup = time_for_loop / time_list_comprehension
print(f"List comprehension is {speedup:.2f}x faster than the traditional for loop.")


Time for traditional for loop: 27.587440 seconds
Time for list comprehension: 25.193669 seconds
List comprehension is 1.10x faster than the traditional for loop.


In [5]:
import timeit

# Traditional for loop approach with a more complex operation
def complex_operation_for_loop(n):
    result = []
    for x in range(n):
        value = (x ** 2 + 10) / 3.14  # More complex operation
        result.append(value)
    return result

# List comprehension approach with the same operation
def complex_operation_list_comprehension(n):
    return [(x ** 2 + 10) / 3.14 for x in range(n)]

# Number of elements in the list
n = 10**7

# Time the for loop approach
time_for_loop = timeit.timeit('complex_operation_for_loop(n)', 
                              globals=globals(), 
                              number=10)

# Time the list comprehension approach
time_list_comprehension = timeit.timeit('complex_operation_list_comprehension(n)', 
                                        globals=globals(), 
                                        number=10)

# Print the results
print(f"Time for traditional for loop: {time_for_loop:.6f} seconds")
print(f"Time for list comprehension: {time_list_comprehension:.6f} seconds")

# Compare the speed
speedup = time_for_loop / time_list_comprehension
print(f"List comprehension is {speedup:.2f}x faster than the traditional for loop.")


Time for traditional for loop: 6.842574 seconds
Time for list comprehension: 6.398477 seconds
List comprehension is 1.07x faster than the traditional for loop.


In [27]:
import pandas as pd
import numpy as np
from dependencies.utils import get_index_to_scenario_for_betmap, find_positions, get_values_by_keys, get_scenarios_vectorized_optimized

x = [0]*46 + [1] + [0]*2
y = [0] + [1] + [0]*44 + [1] + [0]*2

df = pd.DataFrame({"BetMap": [x, y]})                 

INDEX_TO_SCENARIO_BET_MAP = get_index_to_scenario_for_betmap()

def get_scenarios(x: list)-> list:
    """Get list of 7x7 soccer scenarios by given list of dummies"""
    
    INDEX_TO_SCENARIO_BET_MAP = get_index_to_scenario_for_betmap()
    
    positions = find_positions(x, target_element=1)
    scenarios = get_values_by_keys(INDEX_TO_SCENARIO_BET_MAP, positions)
    print(scenarios)

    return scenarios

scenario = '0 : 1'
#check_scenario = lambda x: scenario in x
# Check if scenario is inside the BetMap
#df['flag'] = df['BetMap'].apply(get_scenarios).apply(check_scenario)
#df

# Convert BetMap to a NumPy array
betmap_matrix = np.vstack(df['BetMap'].values)

# Get a 2D boolean array for active scenarios and the scenario strings
active_scenarios, scenario_strings = get_scenarios_vectorized_optimized(betmap_matrix)

# Find the index of the target scenario in the scenario_strings array
scenario_index = np.where(scenario_strings == scenario)[0][0]

# Vectorized check: Get the flag for each row where the scenario is active
scenario_flags = active_scenarios[:, scenario_index]

print((active_scenarios, scenario_strings))
scenario_index, scenario_flags

(array([[False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False,  True, False, False],
       [False,  True, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False,  True, False, False]]), array(['0 : 0', '0 : 1', '0 : 2', '0 : 3', '0 : 4', '0 : 5', '0 : 6',
       '1 : 0', '1 : 1', '1 : 2', '1 : 3', '1 : 4', '1 : 5', '1 : 6',
       '2 : 0', '2 : 1', '2 : 2', '2 : 3', '2 : 4', '2 : 5', '2 : 6',
   

(1, array([False,  True]))

In [ ]:

@profile
def get_bet_return(df: pd.DataFrame, allocation_array: list, scenario: str) -> float:
    """Get financial return of the bet by given allocation and scenario"""
    check_scenario = lambda x: scenario in x
    # Check if scenario is inside the BetMap
    df['flag'] = df['BetMap'].apply(get_scenarios).apply(check_scenario)
    
    logger.info(f"Bets won:\n{df[df.flag][['Market', 'Bet', 'Scenario', 'Odd', 'flag']]}")
    
    logger.info(f"Allocation won:\n{pd.Series(allocation_array)[df.flag.to_list()]}")
    
    # Calculate the financial return
    return sum(df['Odd'] * df['flag'] * allocation_array)
